# FanGraphs fWAR Leaderboard — Through May 3 (2010–present)

Downloads the FanGraphs batting fWAR leaderboard for each season filtered to **Mar 1 → May 3** using `pybaseball`.

### Install deps (first time only)
```bash
pip install pybaseball pandas
```

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pybaseball', 'pandas', '-q'])

0

In [ ]:
import time
import requests
import pandas as pd
from datetime import datetime
from pybaseball import cache

cache.enable()

START_YEAR  = 2010
END_YEAR    = datetime.today().year
OUTPUT_FILE = 'fwar_thru_may3.csv'
SLEEP_SEC   = 2

FG_LEGACY = 'https://www.fangraphs.com/leaders/legacy/leader-stats-range.aspx'

session = requests.Session()
session.headers.update({
    'User-Agent':      'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Accept':          'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer':         'https://www.fangraphs.com/',
})

print('Ready.')

In [ ]:
def fetch_season(year: int) -> pd.DataFrame:
    resp = session.get(FG_LEGACY, params={
        'startdate': f'{year}-03-01',
        'enddate':   f'{year}-05-03',
        'pos': 'all', 'stats': 'bat', 'lg': 'all', 'qual': 'y',
    }, timeout=20)
    resp.raise_for_status()

    tables = pd.read_html(resp.text)

    # Find whichever table has a 'Name' column — insulates us from page layout changes
    for t in tables:
        if 'Name' in t.columns and len(t) > 5:
            return t

    # Nothing matched — print all table shapes so we can debug
    print(f'    [{year}] {len(tables)} tables found, none had a Name column:')
    for i, t in enumerate(tables):
        print(f'      [{i}] shape={t.shape} cols={list(t.columns[:4])}')
    return pd.DataFrame()


all_frames = []
print(f'Downloading {START_YEAR}–{END_YEAR} (Mar 1 → May 3)...\n')

for year in range(START_YEAR, END_YEAR + 1):
    try:
        df = fetch_season(year)
        if df.empty:
            print(f'  {year}: no data')
            continue
        df.insert(0, 'Season', year)
        print(f'  {year}: {len(df)} rows')
        all_frames.append(df)
    except Exception as e:
        print(f'  {year}: error — {e}')
    time.sleep(SLEEP_SEC)

print('\nDone.')
if all_frames:
    print('Columns:', list(all_frames[0].columns))

In [ ]:
combined = pd.concat(all_frames, ignore_index=True)

print('Columns:', list(combined.columns))

# Normalise WAR column name — pybaseball may call it WAR, fWAR, or omit it
war_candidates = [c for c in combined.columns if c.strip().upper() in ('WAR', 'FWAR')]
if war_candidates:
    combined.rename(columns={war_candidates[0]: 'fWAR'}, inplace=True)
    war_col = 'fWAR'
else:
    wrc_candidates = [c for c in combined.columns if 'wrc' in c.lower()]
    war_col = wrc_candidates[0] if wrc_candidates else combined.columns[-1]
    print(f'NOTE: no WAR column found — sorting by {war_col!r} instead')

combined.sort_values(['Season', war_col], ascending=[True, False], inplace=True)
combined.reset_index(drop=True, inplace=True)

combined.to_csv(OUTPUT_FILE, index=False)
print(f'Saved {len(combined)} rows x {len(combined.columns)} cols -> {OUTPUT_FILE}')
combined.head(10)

## Top 5 fWAR leaders — Mar 1 through May 3, by season

In [ ]:
display_cols = [c for c in ['Season', 'Name', 'Team', 'G', 'PA', war_col, 'wRC+', 'HR', 'AVG', 'OBP', 'SLG'] if c in combined.columns]

top5 = (
    combined
    .sort_values(['Season', war_col], ascending=[True, False])
    .groupby('Season', sort=False)
    .head(5)
    .reset_index(drop=True)
)[display_cols]

print(f'Top 5 by {war_col} (Mar 1 – May 3), {combined["Season"].min()}–{combined["Season"].max()}')
display(top5)

## Yankees players in the top-5 fWAR — Mar 1 through May 3

In [ ]:
yankees = top5[top5['Team'] == 'NYY'].reset_index(drop=True)

print(f'Yankees appearances in the top-5 (Mar 1 – May 3): {len(yankees)} across {yankees["Season"].nunique()} season(s)')
print()
display(yankees)

# Seasons with no Yankee in the top 5
all_seasons = set(combined['Season'].unique())
yankee_seasons = set(yankees['Season'].unique())
no_yankee = sorted(all_seasons - yankee_seasons)
print(f'\nSeasons with NO Yankee in the top 5: {no_yankee}')